<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/RAG_with_New_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [168]:
!pip install -q huggingface_hub langchain_core langchain-deepseek langchain_huggingface torch tiktoken datasets "chonkie[semantic]" chonkie chromadb "bitsandbytes>=0.46.1"

In [167]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import login, whoami
from google.colab import userdata
import os


# Colab Secrets (🔑 in the sidebar)
os.environ["hf_token"] = userdata.get("HF_TOKEN")
os.environ["deepseek_api_key"] = userdata.get("DEEPSEEK_API_KEY")


login(token=os.environ["hf_token"])
print(whoami())

In [169]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_v3.1"

dataset_dict = load_dataset(repo_id, token=os.environ["hf_token"])
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()
masked_df = dataset_dict["masked_test"].to_pandas()

DISEASE_LABELS = sorted(train_df["target_disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(masked_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

train cases: 445 | test cases: 73 | diseases: 13


,case_id,pair_id,split,target_disease,gender,age,raw_case_text,soap_note
0,PMC9233216_01,PMC9233216_01::Brucellosis,train,Brucellosis,Female,11,"The patient is an 11-year-old Lebanese girl, d...",{'subjective': {'chief_complaint': 'An 11-year...
1,PMC9353035_01,PMC9353035_01::Brucellosis,train,Brucellosis,Male,40,A 40-year-old male farmer without any underlyi...,{'subjective': {'chief_complaint': 'A 40-year-...
2,PMC8710768_01,PMC8710768_01::Brucellosis,train,Brucellosis,Male,67,"A 67-year-old man, a local farmer specializing...",{'subjective': {'chief_complaint': 'A 67-year-...
3,PMC9389076_01,PMC9389076_01::Brucellosis,train,Brucellosis,Male,42,A 42-year-old man was admitted to our hospital...,{'subjective': {'chief_complaint': 'A 42-year-...
4,PMC4154277_01,PMC4154277_01::Brucellosis,train,Brucellosis,Female,25,A 25 year old woman was referred with a histor...,{'subjective': {'chief_complaint': 'A 25-year-...


## **Parse Soap Notes**

In [170]:
import json

def parse_soap_sections(note):
    def render(key):
        values = (" ".join(str(v).split()) for v in (note.get(key) or {}).values() if v)
        return " ".join(values).replace("et al.", "et al")  # stop "et al." from being treated as a sentence ending
    return render("subjective"), render("objective"), render("assessment_and_plan")

## **Chunking Strategies**

In [171]:
from chonkie import (
    SemanticChunker,
    RecursiveChunker,
    RecursiveRules,
    RecursiveLevel,
)

import re

# ---------- Chunkers ----------
def perform_semantic_chunking(text, threshold=0.7, chunk_size=500, similarity_window=3, min_sentences_per_chunk=2, skip_window=0, batch=False):

    semantic_chunker = SemanticChunker(
        threshold=threshold,
        chunk_size=chunk_size,              # tokens
        similarity_window=similarity_window,
        min_sentences_per_chunk=min_sentences_per_chunk,
        skip_window=skip_window,
    )

    if batch:
        return [semantic_chunker.chunk(t) for t in text]    # text is a list of strings
    return semantic_chunker.chunk(text)



def perform_structured_chunking(text, chunk_size=500, batch=False):

    rules = RecursiveRules(levels=[
      RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
      RecursiveLevel(delimiters=["\n"], include_delim="prev"),
      RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    ])

    recursive_chunker = RecursiveChunker(
      tokenizer="character",
      chunk_size=chunk_size,              # characters
      rules=rules,
    )

    if batch:
        return [recursive_chunker.chunk(t) for t in text]    # text is a list of strings
    return recursive_chunker.chunk(text)

## **ChromaDB and Vector Database**

**Build `ChromaDB` connection on-demand**

In [172]:
import chromadb
client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

def make_records(df=train_df, chunk_fn=None):
    records = []
    for row in df.itertuples():
        # Extract soap note
        subjective, objective, assessment_and_plan = parse_soap_sections(row.soap_note)

        # chunk_fn returns one list of chunks per section; flatten them into a single list
        chunks = [c for section in chunk_fn([objective, assessment_and_plan]) for c in section]

        for chunk_id, chunk in enumerate(chunks):
            chunk_text = chunk.text if hasattr(chunk, "text") else str(chunk)
            records.append({
                "document": chunk_text,
                "metadata": {
                    "case_id": row.case_id,
                    "disease": row.target_disease
                },
            })
    return records


def build_collection(name, chunk_fn=None, REBUILD=False):
    if chunk_fn is None:
        raise ValueError("chunk_fn must be provided")

    # REBUILD = False, reuse existing connection
    if not REBUILD:
       try:
          collection = client.get_collection(name)
          print("Resuing existing collection!")
          return collection
       except Exception:
          print("Collection not found or corrupt!")
          return None

    # records: list of {"document": str, "metadata": dict}

    records = make_records(chunk_fn=chunk_fn)

    try:
        client.delete_collection(name)
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"}
    )

    batch = client.get_max_batch_size()
    for start in range(0, len(records), batch):
        part = records[start:start + batch]
        collection.upsert(
            ids=[f"chunk_{start + i}" for i in range(len(part))],
            documents=[r["document"] for r in part],
            metadatas=[r["metadata"] for r in part],
        )

    print(f"{name}: built {collection.count()} chunks")
    return collection

In [173]:
CHROMA_CONFIG = {"name": "semantic_chunking_soap", "chunk_fn": lambda texts: perform_semantic_chunking(texts, batch=True)}

CHROMA_COLLECTION = build_collection(name=CHROMA_CONFIG["name"], chunk_fn=CHROMA_CONFIG["chunk_fn"])

Resuing existing collection!


## **RAG Pipeline**

**Plumbing functions (Handles retrieval and generate)**

In [174]:
def _retrieve(collection, query, k, search_k=20):
    n_available = collection.count()        # Count the number of chunks in collection

    if n_available == 0:
        return []

    results = collection.query(             # Query related chunks, including "documents", "metadata" and "distances"
        query_texts=[query],
        n_results=min(search_k, n_available),
        include=["documents", "metadatas", "distances"],
    )

    best_by_case = {}                       # Dict to store top-k chunks

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        case_id = metadata["case_id"]
        distance = float(distance)

        hit = {
            "case_id": case_id,
            "disease": metadata["disease"],
            "distance": distance,
            "text": document,
        }

        # Keep the closest chunk for each case.
        existing = best_by_case.get(case_id)

        if existing is None or distance < existing["distance"]:
            best_by_case[case_id] = hit

    return sorted(best_by_case.values(), key= lambda result: result["distance"])[:k]

**Qwen 3 As Small Model**

In [175]:
import torch
# from transformers import AutoProcessor, AutoModelForMultimodalLM,
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


MODEL_NAME = "Qwen/Qwen3-4B"

# load the tokenizer and the model
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)
MODEL = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

def _generate(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )

    model_inputs = TOKENIZER([text], return_tensors="pt").to(MODEL.device)

    # conduct text completion
    generated_ids = MODEL.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    content = TOKENIZER.decode(output_ids, skip_special_tokens=True).strip("\n")

    return content

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

**Realize system prompt**

In [184]:
TOP_K_DIAGNOSES = 3

SYS_PROMPT_NO_RAG = (
f"""
You are an expert clinical assistant.\n
You will be given a medical CASE PRESENTATION. You MUST read through this presentation carefully and determine the {TOP_K_DIAGNOSES} most likely diagnoses for this patient.\n

INSTRUCTIONS:\n
1. You MUST output as your answer a numbered list of {TOP_K_DIAGNOSES} diseases, from most likely to least likely for the given case presentation.\n
2. You MUST respond for each disease with exactly ONE (1) line, formatted as '<rank> <disease>'.\n
3. YOU MUST only respond with diseases from this list, whose names MUST be copied verbatim:\n
"""
+ "\n".join(f"\t- {name}" for name in DISEASE_LABELS)
)

SYS_PROMPT_WITH_RAG = f"""{SYS_PROMPT_NO_RAG}
\nEXTRA CLUES:
- Use ONLY the retrieved case reports below as evidence (DO NOT diagnose them, only diagnose CASE PRESENTATION):\n"""

**Diagnose**

In [185]:
def diagnose(question, k=TOP_K_DIAGNOSES, rag=True):
    # Retrive top-k chunks based on question (query)
    retrieved = _retrieve(
        collection=CHROMA_COLLECTION,
        query=question,
        k=k,
        search_k=max(20, k * 4),
    )

    # Format retrieved chunks into usable prompts
    context = "\n\n".join(
        f"Case ({result['disease']}):\n"
        f"{result['text']}"
        for result in retrieved
    )

    user_prompt = (
        f"{context}\n\n"
        f"CASE PRESENTATION (TO BE DIAGNOSED):\n{question}"
    )

    # Determine whether to use RAG mode or not
    sys_prompt = SYS_PROMPT_WITH_RAG if rag else SYS_PROMPT_NO_RAG

    # Generate response with chunks' aid
    response = _generate(system_prompt=sys_prompt, user_prompt=user_prompt)

    return response, context

## **Benchmark and evaluation**

**Single evaluation function**

In [186]:
def evaluate_retrieval_by_disease(eval_df, k=TOP_K_DIAGNOSES):
    """Test the retrieval stage independently, overall and per disease.

    Returns
    -------
    overall    : dict      micro-average over all cases + macro-average over diseases
    by_disease : DataFrame one row per disease (n, recall@k, precision@k, top1_accuracy, mrr@k)
    cases      : DataFrame one row per test case, incl. the retrieved labels (for error analysis)
    """
    records = []

    for row in eval_df.itertuples():
        # Extract soap note and form query (no assement to prevent disease leakage)
        subjective, objective, assessment_and_plan = parse_soap_sections(row.soap_note)
        query = f"""SUBJECTIVE ASSESSMENT\n{subjective}\n\nOBJECTIVE ASSESSMENT\n{objective}\n"""

        # Retrieval
        retrieved = [
            result["disease"]
            for result in _retrieve(collection=CHROMA_COLLECTION, query=query, k=k)
        ]
        matches = [d == row.target_disease for d in retrieved]
        rank = matches.index(True) + 1 if any(matches) else None

        records.append({
            "case_id": getattr(row, "case_id", row.Index),
            "disease": row.target_disease,
            "hit": any(matches),                                   # recall@k (per case)
            "top1": bool(matches) and matches[0],                  # top-1 accuracy (per case)
            "precision": sum(matches) / len(matches) if matches else 0.0,
            "rr": 1 / rank if rank else 0.0,                       # reciprocal rank
            "retrieved": retrieved,
        })

    cases = pd.DataFrame(records)

    by_disease = (
        cases.groupby("disease")
        .agg(**{
            "n": ("hit", "size"),
            f"recall@{k}": ("hit", "mean"),
            f"precision@{k}": ("precision", "mean"),
            "top1_accuracy": ("top1", "mean"),
            f"mrr@{k}": ("rr", "mean"),
        })
        .sort_values([f"recall@{k}", "top1_accuracy"])             # weakest diseases first
    )

    metric_cols = [c for c in by_disease.columns if c != "n"]
    overall = {
        "k": k,
        "n": len(cases),
        # micro: every case weighted equally
        "micro": {
            f"recall@{k}": cases["hit"].mean(),
            f"precision@{k}": cases["precision"].mean(),
            "top1_accuracy": cases["top1"].mean(),
            f"mrr@{k}": cases["rr"].mean(),
        },
        # macro: every disease weighted equally
        "macro": by_disease[metric_cols].mean().to_dict(),
    }
    return overall, by_disease.round(3), cases


In [ ]:
def sample_train_subset(train_df, frac=0.10, seed=42, stratify=False):
    """Randomly select ~frac of train_df; stratify=True takes the same fraction per disease (min. 1)."""
    if not stratify:
        return train_df.sample(frac=frac, random_state=seed)
    return pd.concat([
        g.sample(n=max(1, round(frac * len(g))), random_state=seed)
        for _, g in train_df.groupby("disease")
    ])

subset_df = sample_train_subset(train_df, frac=0.10, seed=42)
evaluate_retrieval_by_disease(subset_df)

**RAG vs. No-RAG**

In [187]:
import json
import os

import pandas as pd
from tqdm.auto import tqdm


def run_config(k, use_rag=True):
    config = {
        "llm": MODEL_NAME,
        "top_k_diagnoses": TOP_K_DIAGNOSES,
    }

    if use_rag:
        config.update(
            vector_store="ChromaDB",
            collection_name=CHROMA_CONFIG["name"],
            n_indexed_chunks=CHROMA_COLLECTION.count(),
            retrieve_k=k,
        )

    return config


def load_predictions(path):
    if not os.path.exists(path):
        return pd.DataFrame()

    with open(path, encoding="utf-8") as f:
        records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    return pd.DataFrame(records)

def run_test_predictions(eval_df, path, k=TOP_K_DIAGNOSES, n_per_disease=None, seed=42, use_rag=True):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    if n_per_disease is None:
        cases = eval_df
    else:
        cases = pd.concat(
            [g.sample(n=min(n_per_disease, len(g)), random_state=seed) for _, g in eval_df.groupby("disease")],
            ignore_index=True,
        )

    # Stored on every record; the chunking function isn't JSON-serializable, so skip it
    config = {key: value for key, value in run_config(k=k, use_rag=use_rag).items() if key != "chunking_strategy"}

    print(f"Running {len(cases)} cases -> {path}")

    with open(path, "w", encoding="utf-8") as f:   # "w" overwrites any previous run
        for row in tqdm(cases.itertuples(), total=len(cases), desc="Diagnosing" if use_rag else "Diagnosing (no RAG)"):
            subjective, objective, assessment_and_plan = parse_soap_sections(row.soap_note)
            query = f"""SUBJECTIVE ASSESSMENT\n{subjective}\n\nOBJECTIVE ASSESSMENT\n{objective}\n"""

            raw_predictions, _ = diagnose(question=query, k=k, rag=use_rag)
            record = {
                "id": row.case_id,
                "true_disease": row.target_disease,
                "raw_prediction": raw_predictions,
                "config": config,
            }
            f.write(json.dumps(record) + "\n")
            f.flush()   # keep partial results if the run crashes

    return load_predictions(path)


# def run_test_predictions(
#     eval_df,
#     path,
#     mode="resume",
#     k=5,
#     n_per_disease=None,
#     seed=42,
#     use_rag=True,
# ):
#     if mode not in ("resume", "force"):
#         raise ValueError("mode must be 'resume' or 'force'")

#     os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

#     # In "force" mode, remove the ".json" file and create a new one
#     if mode == "force" and os.path.exists(path):
#         os.remove(path)


#     config = run_config(k=k, use_rag=use_rag)

#     existing = load_predictions(path)

#     # Do not mix predictions from different configurations. ("resume" mode only)

#     if len(existing) > 0:
#         old_configs = {
#             json.dumps(c, sort_keys=True)
#             for c in existing["config"]
#         }

#         current_config = json.dumps(config, sort_keys=True)

#         if old_configs != {current_config}:
#             raise ValueError(
#                 f"{path} was produced with different settings. "
#                 "Use mode='force' to overwrite it, "
#                 "or choose a different output path."
#             )

#     # Track case_ids that have been tested
#     done_ids = set(existing["case_id"]) if len(existing) > 0 else set()

#     if n_per_disease is None:
#         wanted = eval_df.copy()
#     else:
#         wanted = pd.concat(
#             [
#                 group.sample(
#                     n=min(n_per_disease, len(group)),
#                     random_state=seed,
#                 )
#                 for _, group in eval_df.groupby("disease")
#             ],
#             ignore_index=True,
#         )

#     todo = wanted[~wanted["case_id"].isin(done_ids)]

#     print(
#         f"{len(wanted)} cases wanted | "
#         f"{len(wanted) - len(todo)} already in {path} | "
#         f"running {len(todo)} new"
#     )

#     # Predict_fn changes based on RAG flag
#     predict_fn = lambda text: diagnose(question=text, k=k, rag=use_rag)


#     with open(path, "a", encoding="utf-8") as f:
#         for row in tqdm(
#             todo.itertuples(),
#             total=len(todo),
#             desc="Diagnosing" if use_rag else "Diagnosing (no RAG)",
#         ):


#             raw_prediction, retrieved = predict_fn(row.case_text)

#             # Basic fields of record
#             record = {
#                 "id": row.case_id,
#                 "true_disease": row.disease,
#                 "raw_prediction": raw_prediction,
#                 "config": {
#                     key: value
#                     for key, value in config.items()
#                     if key != "chunking_strategy"   # "chunking_strategy" is a function, don't record it
#                 },
#             }
#             # Write into JSON file
#             f.write(json.dumps(record) + "\n")
#             f.flush()

#     return load_predictions(path)

In [181]:
RESULTS_DIR = "/content/drive/MyDrive/colab_projects/VGU_WS26_Project/RAG_Evaluation"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Compare RAG and NO RAG retrieval
RAG_EVAL_JSON = os.path.join(RESULTS_DIR, "test_predictions.jsonl")
NO_RAG_EVAL_JSON = os.path.join(RESULTS_DIR, "test_predictions_no_rag.jsonl")


RUN_MODE = "force"    # "resume": only run cases missing from the file | "force": delete the file and redo everything
N_PER_DISEASE = None   # None: every test case | 1: a quick run with one case per disease

In [188]:
predictions_df = run_test_predictions(eval_df=masked_df, path=RAG_EVAL_JSON)


Running 73 cases -> /content/drive/MyDrive/colab_projects/VGU_WS26_Project/RAG_Evaluation/test_predictions.jsonl


Diagnosing:   0%|          | 0/73 [00:00<?, ?it/s]

In [189]:
baseline_predictions_df = run_test_predictions(eval_df=masked_df, path=NO_RAG_EVAL_JSON, use_rag=False)

Running 73 cases -> /content/drive/MyDrive/colab_projects/VGU_WS26_Project/RAG_Evaluation/test_predictions_no_rag.jsonl


Diagnosing (no RAG):   0%|          | 0/73 [00:00<?, ?it/s]

## **Scoring predictions and benchmarks**

`top1_accuracy`: Fraction of cases where the first diagnosis is correct

`hit_at_k`	   : Fraction where the true disease appears in Gemma's top five diagnoses

`mrr`: Average reciprocal rank of the true disease in Gemma's predictions

`retrieval_recall`: Fraction of cases where Chroma retrieved at least one case with the true disease label

`top1_grounded`:	Fraction where Gemma's first diagnosis label appears among the retrieved disease labels


In [196]:
def parse_diagnoses(prediction, labels=DISEASE_LABELS, top_k=TOP_K_DIAGNOSES):
    """Return up to top_k labels found in `prediction`, ordered by where each first appears
    in the text (not by iterating `labels`) -- so the order reflects the model's own ranking,
    even if it didn't follow the "<rank>. <disease name>" format exactly."""
    prediction_lower = prediction.lower()
    found = [(prediction_lower.find(label.lower()), label) for label in labels]
    found = [(idx, label) for idx, label in found if idx != -1]
    found.sort(key=lambda pair: pair[0])
    return [label for _, label in found][:top_k]

def score_predictions(predictions_df, top_k=TOP_K_DIAGNOSES):
    # Make a copy of predictions dataframe
    df = predictions_df.copy()

    # RAG-specific fields may be absent in no-RAG prediction files. Append them with NULL as value.
    retrieval_fields = [
        "retrieved_ids",
        "retrieved_diseases",
        "retrieved_chunk_ids",
        "retrieved_distances",
    ]

    for field in retrieval_fields:
        if field not in df.columns:
            df[field] = [[] for _ in range(len(df))]
        else:
            df[field] = df[field].apply(
                lambda x: x if isinstance(x, list) else []
            )

    # Reformat diseases in "raw_prediction" to valid data structure
    df["predicted_diagnoses"] = df["raw_prediction"].apply(lambda text: parse_diagnoses(text, top_k=top_k))

    # Introduce evaluation metrics to dataset
    ranks = [
        predicted.index(true) + 1 if true in predicted else None
        for predicted, true in zip(df["predicted_diagnoses"], df["true_disease"])
    ]
    df["top1_correct"] = [bool(p) and p[0] == t for p, t in zip(df["predicted_diagnoses"], df["true_disease"])]
    df["hit_at_k"] = [rank is not None for rank in ranks]
    df["reciprocal_rank"] = [1 / rank if rank else 0.0 for rank in ranks]
    df["true_in_retrieved"] = [t in r for t, r in zip(df["true_disease"], df["retrieved_diseases"])]
    df["top1_grounded"] = [bool(p) and p[0] in r for p, r in zip(df["predicted_diagnoses"], df["retrieved_diseases"])]
    return df


def summarize(scored_df):
    retrieved = scored_df[scored_df["true_in_retrieved"]]
    missed = scored_df[~scored_df["true_in_retrieved"]]
    return {
        "n": len(scored_df),
        "top1_accuracy": float(scored_df["top1_correct"].mean()),
        "hit_at_k": float(scored_df["hit_at_k"].mean()),
        "mrr": float(scored_df["reciprocal_rank"].mean()),
        "retrieval_recall": float(scored_df["true_in_retrieved"].mean()),
        "top1_grounded": float(scored_df["top1_grounded"].mean()),
        "hit_at_k_if_retrieved": float(retrieved["hit_at_k"].mean()) if len(retrieved) else float("nan"),
        "hit_at_k_if_not_retrieved": float(missed["hit_at_k"].mean()) if len(missed) else float("nan"),
    }

**Print out metrics **

In [191]:
predictions_df = load_predictions(RAG_EVAL_JSON)  # read the saved run (re-score without calling Gemma)
scored_df = score_predictions(predictions_df)
metrics = summarize(scored_df)
print({name: round(value, 3) if isinstance(value, float) else value for name, value in metrics.items()})

{'n': 73, 'top1_accuracy': 0.37, 'hit_at_k': 0.534, 'mrr': 0.445, 'retrieval_recall': 0.0, 'top1_grounded': 0.0, 'hit_at_k_if_retrieved': nan, 'hit_at_k_if_not_retrieved': 0.534}


In [192]:
scored_df[["id", "true_disease", "predicted_diagnoses", "top1_correct", "hit_at_k", "true_in_retrieved"]]

,id,true_disease,predicted_diagnoses,top1_correct,hit_at_k,true_in_retrieved
0,PMC9892713_01,Brucellosis,"[Q Fever, Extrapulmonary Tuberculosis, Pulmona...",False,False,False
1,PMC9767972_01,Brucellosis,"[Brucellosis, Q Fever, Extrapulmonary Tubercul...",True,True,False
2,PMC6639843_01,Brucellosis,"[Brucellosis, Leptospirosis, Q Fever]",True,True,False
3,PMC10921472_01,Brucellosis,"[Pulmonary Tuberculosis, Extrapulmonary Tuberc...",False,False,False
4,PMC11746083_04,Brucellosis,"[Extrapulmonary Tuberculosis, Pulmonary Tuberc...",False,True,False
...,...,...,...,...,...,...
68,PMC12858201_01,Pulmonary Tuberculosis,"[Extrapulmonary Tuberculosis, Pulmonary Tuberc...",False,True,False
69,PMC9112304_01,Q Fever,"[Pulmonary Tuberculosis, Extrapulmonary Tuberc...",False,False,False
70,PMC5803979_03,Q Fever,"[Pulmonary Tuberculosis, Extrapulmonary Tuberc...",False,True,False
71,PMC11165120_01,Q Fever,"[Pulmonary Tuberculosis, Extrapulmonary Tuberc...",False,False,False


In [193]:
def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = successes / n
    denominator = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denominator
    half_width = z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5 / denominator
    return centre - half_width, centre + half_width


def per_disease_report(scored_df):
    rows = []
    for disease, cases in scored_df.groupby("true_disease"):
        n = len(cases)
        hits = int(cases["hit_at_k"].sum())
        ci_low, ci_high = wilson_interval(hits, n)

        answered_first = scored_df[scored_df["predicted_diagnoses"].apply(lambda p: bool(p) and p[0] == disease)]
        rows.append({
            "disease": disease,
            "n": n,
            "top1_rate": cases["top1_correct"].mean(),
            "hit_at_k_rate": hits / n,
            "hit_at_k_ci_low": ci_low,
            "hit_at_k_ci_high": ci_high,
            "retrieval_recall": cases["true_in_retrieved"].mean(),
            "top1_precision": (answered_first["true_disease"] == disease).mean() if len(answered_first) else float("nan"),
        })
    return pd.DataFrame(rows).set_index("disease").sort_values("hit_at_k_rate")


per_disease = per_disease_report(scored_df)
print(
    f"Macro (equal weight per disease) | top-1: {per_disease['top1_rate'].mean():.1%} | "
    f"hit@{TOP_K_DIAGNOSES}: {per_disease['hit_at_k_rate'].mean():.1%} | "
    f"retrieval recall: {per_disease['retrieval_recall'].mean():.1%}"
)
per_disease.round(2)

Macro (equal weight per disease) | top-1: 33.4% | hit@3: 50.2% | retrieval recall: 0.0%


,n,top1_rate,hit_at_k_rate,hit_at_k_ci_low,hit_at_k_ci_high,retrieval_recall,top1_precision
disease,,,,,,,
Malaria,5,0.00,0.00,-0.00,0.43,0.0,NaN
Hepatitis C,5,0.20,0.20,0.04,0.62,0.0,0.25
Melioidosis,5,0.00,0.20,0.04,0.62,0.0,NaN
Leptospirosis,4,0.25,0.25,0.05,0.70,0.0,0.33
Q Fever,4,0.00,0.25,0.05,0.70,0.0,0.00
Hepatitis B,7,0.29,0.43,0.16,0.75,0.0,1.00
Neurocysticercosis,5,0.40,0.60,0.23,0.88,0.0,0.67
Leprosy,5,0.40,0.60,0.23,0.88,0.0,0.67
Brucellosis,6,0.50,0.67,0.30,0.90,0.0,0.50


In [195]:
import math


def load_scored(path):
    predictions = load_predictions(path)
    if predictions.empty:
        raise FileNotFoundError(f"No predictions in {path} -- run the cell that creates it first.")
    return score_predictions(predictions)


def exact_mcnemar_p(only_rag_right, only_no_rag_right):
    n = only_rag_right + only_no_rag_right
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(only_rag_right, only_no_rag_right) + 1)) / 2**n
    return min(1.0, 2 * tail)


def compare_rag_vs_baseline(rag_scored, base_scored):
    merged = rag_scored.merge(base_scored, on="id", suffixes=("_rag", "_base"))

    rows = []
    for metric in ["top1_correct", "hit_at_k"]:
        rag, base = merged[f"{metric}_rag"], merged[f"{metric}_base"]
        only_rag, only_base = int((rag & ~base).sum()), int((~rag & base).sum())
        rows.append({
            "metric": metric,
            "rag": float(rag.mean()),
            "no_rag": float(base.mean()),
            "difference": float(rag.mean() - base.mean()),
            "only_rag_right": only_rag,
            "only_no_rag_right": only_base,
            "p_value": exact_mcnemar_p(only_rag, only_base),
        })
    mrr_rag, mrr_base = merged["reciprocal_rank_rag"].mean(), merged["reciprocal_rank_base"].mean()
    rows.append({"metric": "mrr", "rag": float(mrr_rag), "no_rag": float(mrr_base), "difference": float(mrr_rag - mrr_base)})
    return merged, pd.DataFrame(rows).set_index("metric")


def per_disease_comparison(merged):
    by_disease = merged.groupby("true_disease_rag")
    comparison = pd.DataFrame({
        "n": by_disease.size(),
        "hit_at_k_rag": by_disease["hit_at_k_rag"].mean(),
        "hit_at_k_no_rag": by_disease["hit_at_k_base"].mean(),
        "top1_rag": by_disease["top1_correct_rag"].mean(),
        "top1_no_rag": by_disease["top1_correct_base"].mean(),
    })
    comparison["hit_at_k_gain"] = comparison["hit_at_k_rag"] - comparison["hit_at_k_no_rag"]
    comparison["top1_gain"] = comparison["top1_rag"] - comparison["top1_no_rag"]
    return comparison.sort_values("top1_gain")


rag_scored = load_scored(RAG_EVAL_JSON)
base_scored = load_scored(NO_RAG_EVAL_JSON)
merged, comparison = compare_rag_vs_baseline(rag_scored, base_scored)

print(f"Compared on {len(merged)} cases that appear in both runs")
display(comparison.round(3))
per_disease_comparison(merged).round(2)


Compared on 73 cases that appear in both runs


,rag,no_rag,difference,only_rag_right,only_no_rag_right,p_value
metric,,,,,,
top1_correct,0.370,0.301,0.068,6.0,1.0,0.125
hit_at_k,0.534,0.521,0.014,4.0,3.0,1.000
mrr,0.445,0.393,0.053,NaN,NaN,NaN


,n,hit_at_k_rag,hit_at_k_no_rag,top1_rag,top1_no_rag,hit_at_k_gain,top1_gain
true_disease_rag,,,,,,,
Brucellosis,6,0.67,0.67,0.50,0.50,0.00,0.00
Extrapulmonary Tuberculosis,5,0.80,1.00,0.20,0.20,-0.20,0.00
Leptospirosis,4,0.25,0.25,0.25,0.25,0.00,0.00
Hepatitis B,7,0.43,0.29,0.29,0.29,0.14,0.00
Q Fever,4,0.25,0.25,0.00,0.00,0.00,0.00
Pulmonary Tuberculosis,5,1.00,1.00,0.80,0.80,0.00,0.00
Melioidosis,5,0.20,0.20,0.00,0.00,0.00,0.00
Malaria,5,0.00,0.40,0.00,0.00,-0.40,0.00
COVID-19 Virus Disease,9,0.78,0.78,0.56,0.44,0.00,0.11
